<a href="https://colab.research.google.com/github/anishkr-sahu/Genai/blob/main/Huggingface_day_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Wed Aug  5 03:47:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# install transformers
!pip install transformers[sentencepiece] datasets rouge_score -q

  Preparing metadata (setup.py) ... done


In [5]:
# update the package
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 37.6 MB/s eta 0:00:00


In [11]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
from datasets import load_dataset
import matplotlib.pyplot as plt

import pandas as pd

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import nltk
from nltk.tokenize import sent_tokenize
nltk.download("punkt")

from tqdm import tqdm
import torch

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [16]:
!pip install evaluate -q

In [17]:
# which device

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [18]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [19]:
# model
model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

In [20]:
# model used by device
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.28GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.28GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [22]:
dataset_samsum = load_dataset("spencer/samsum_reformat")


dataset_infos.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  923kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  968kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14732 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

In [24]:
dataset_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'sentences', 'sentence_id', 'dialog_id'],
        num_rows: 14732
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'sentences', 'sentence_id', 'dialog_id'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'sentences', 'sentence_id', 'dialog_id'],
        num_rows: 819
    })
})

In [25]:
dataset_samsum['train']['dialogue'][1]

'Olivia: Who are you voting for in this election? \r\nOliver: Liberals as always.\r\nOlivia: Me too!!\r\nOliver: Great'

In [27]:
dataset_samsum['train']['summary'][1]

'B and Olivier are voting for liberals in this election. '

In [28]:
# vector representation
def convert_examples_to_feature(example_batch):
  input_encodings = tokenizer(example_batch['dialogue'], max_length = 1024, truncation = True)

  with tokenizer.as_target_tokenizer():
    target_encodings = tokenizer(example_batch['summary'], max_length = 128, truncation = True)

  return {
      'input_ids': input_encodings['input_ids'],
      'attention_mask': input_encodings['attention_mask'],
      'labels': target_encodings['inputs']
  }

In [31]:
# The 'as_target_tokenizer' method has been deprecated.
# Redefining convert_examples_to_feature to use 'text_target' argument for tokenizing summaries.
# This redefinition is placed here to adhere to the strict instruction to modify ONLY this cell.
def convert_examples_to_feature(example_batch):
  input_encodings = tokenizer(example_batch['dialogue'], max_length = 1024, truncation = True)
  target_encodings = tokenizer(text_target=example_batch['summary'], max_length = 128, truncation = True)

  return {
      'input_ids': input_encodings['input_ids'],
      'attention_mask': input_encodings['attention_mask'],
      'labels': target_encodings['input_ids'] # Corrected from 'inputs' to 'input_ids'
  }

dataset_samsum_pt = dataset_samsum.map(convert_examples_to_feature, batched = True)

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [ ]:
dataset_samsum_pt['train']

In [32]:
dataset_samsum_pt['train']['input_ids'][1]

[18038,
 151,
 2632,
 127,
 119,
 6228,
 118,
 115,
 136,
 2974,
 152,
 10463,
 151,
 35884,
 130,
 329,
 107,
 18038,
 151,
 2587,
 314,
 1242,
 10463,
 151,
 1509,
 1]

In [33]:
dataset_samsum_pt['train']['attention_mask'][1]

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [34]:
dataset_samsum_pt['train']['labels'][1]

[596, 111, 34296, 127, 6228, 118, 33195, 115, 136, 2974, 107, 110, 1]

In [36]:
# training

from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [42]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir = 'pegasus-samsum', num_train_epochs = 1, warmup_steps = 500,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    weight_decay=0.01, logging_steps=10,
    save_steps=1e6,
    gradient_accumulation_steps =16
)

In [45]:
trainer = Trainer(
    model = model_pegasus,
    args = trainer_args,
    data_collator = seq2seq_data_collator,
    train_dataset=dataset_samsum_pt['test'],
    eval_dataset=dataset_samsum_pt['validation']
)

In [46]:
trainer.train()

Step,Training Loss
10,63.952325
20,69.156244
30,65.254065
40,64.595862
50,60.350201


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=52, training_loss=63.50656964228703, metrics={'train_runtime': 473.09, 'train_samples_per_second': 1.731, 'train_steps_per_second': 0.11, 'total_flos': 314203859361792.0, 'train_loss': 63.50656964228703, 'epoch': 1.0})

In [54]:
# evaluation

def generate_batch_sized_chunks(list_of_elements,batch_size):
  for i in range(0,len(list_of_elements), batch_size):
    yield list_of_elements[i: i+batch_size]

def calculate_metric_on_test_ds(dataset, metric,model,tokenizer,
                                batch_size=16, device=device,
                                column_text="dialogue", # Updated default to match usage
                                column_summary="summary"): # Updated default to match usage

  article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
  target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size ))

  all_decoded_summaries = []
  all_target_references = []

  for article_batch, target_batch in tqdm(
      zip(article_batches, target_batches), total=len(article_batches)):

      inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                         padding="max_length", return_tensors='pt')

      summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                                 attention_mask=inputs["attention_mask"].to(device),
                                 length_penalty=0.8, num_beams=8, max_length=128
      )
      # Decode the generated summaries (token IDs) for the current batch
      decoded_summaries_batch = tokenizer.batch_decode(summaries, skip_special_tokens=True, clean_up_tokenization_spaces=True)

      all_decoded_summaries.extend(decoded_summaries_batch)
      all_target_references.extend(target_batch)

  # Now add all collected predictions and references to the metric
  metric.add_batch(predictions=all_decoded_summaries, references=all_target_references)

  score = metric.compute()
  return score

In [55]:
import evaluate

rouge_names = ["rouge1","rouge2","rougeL","rougeLsum"]
rouge_metric = evaluate.load('rouge')

In [57]:
score = calculate_metric_on_test_ds(
    dataset_samsum['test'][0:10], rouge_metric, trainer.model, tokenizer, batch_size=2, column_text='dialogue', column_summary='summary'
)

rouge_dict = dict((rn, score[rn]) for rn in rouge_names)
pd.DataFrame(rouge_dict, index = [f'pegasus'])

100%|██████████| 5/5 [00:38<00:00,  7.79s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.147696,0.040754,0.108696,0.109035


In [58]:
# save model
model_pegasus.save_pretrained("pegasus-samsum-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [59]:
# save tokenizer
tokenizer.save_pretrained("tokenizer")

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')

In [60]:
# load
tokenizer = AutoTokenizer.from_pretrained("/content/tokenizer")


In [69]:
# prediction

gen_kwargs = {"length_penalty": 0.8, "num_beams":8, "max_length": 128}

sample_text = dataset_samsum['test'][0]['dialogue']
reference = dataset_samsum['test'][0]['summary']

pipe = pipeline('text-generation', model='pegasus-samsum-model', tokenizer=tokenizer)

print("Dialogue: ")
print(sample_text)

print("\nReference Summary: ")
print(reference)

print("\nModel Summary:")
print(pipe(sample_text, **gen_kwargs)[0]["generated_text"])

Loading weights:   0%|          | 0/419 [00:00<?, ?it/s]

[transformers] This checkpoint seem corrupted. The tied weights mapping for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are absent from the checkpoint, and we could not find another related tied weight for those keys
[transformers] PegasusForCausalLM LOAD REPORT from: pegasus-samsum-model
Key                                                       | Status     | 
----------------------------------------------------------+------------+-
model.encoder.layers.{0...15}.self_attn_layer_norm.weight | UNEXPECTED | 
model.encoder.layers.{0...15}.fc2.weight                  | UNEXPECTED | 
model.encoder.layers.{0...15}.self_attn.k_proj.weight     | UNEXPECTED | 
model.encoder.layers.{0...15}.self_attn.q_proj.bias       | UNEXPECTED | 
model.encoder.layers.{0...15}.fc1.weight                  | UNEXPECTED | 
model.encoder.layers.{0...15}.self_attn.q_proj.weight     | UNEXPECTED | 
model.encoder.layers.{0...15}.self_attn_layer_norm.bias   | UNEXPECTED |

Dialogue: 
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary: 
A needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Model Summary:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye byesaliva86% bulbs Pseudomonas Pseudomonas
